In [1]:
import polars as pl
import os
from sqlalchemy import create_engine
from os import listdir, environ
from openhexa.toolbox.dhis2.dataframe import get_organisation_units, get_data_elements
from openhexa.sdk import workspace, current_run
from openhexa.toolbox.dhis2 import DHIS2


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/opt/conda/lib/python3.13/site-packages/traitlets/traitlets.py", line 632, in get
    value = obj._trait_values[self.name]
            ~~~~~~~~~~~~~~~~~^^^^^^^^^^^
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/opt/conda/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 340, in dispatch_control
    async with self._control_lock:
               ^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/site-packages/traitlets/traitlets.py", line 687, in __get__
    return t.cast(G, self.get(obj, cls))  # the G should encode the Optional
                     ~~~~~~~~^^^^^^^^^^
  File "/opt/conda/lib/python3.13/site-packages/traitlets/traitlets.py", line 649, i

In [2]:
os.getcwd()

'/home/jovyan/workspace/notebooks'

In [3]:
save_in_db = True

In [4]:
con = workspace.dhis2_connection("pbf-mali-blsq")
dhis = DHIS2(con)

## Lets get the ous

In [5]:
ous_dhis2 = get_organisation_units(dhis)

In [6]:
ous_dhis2.head()

id,name,level,opening_date,closed_date,level_1_id,level_1_name,level_2_id,level_2_name,level_3_id,level_3_name,level_4_id,level_4_name,level_5_id,level_5_name,level_6_id,level_6_name,level_7_id,level_7_name,level_8_id,level_8_name,geometry
str,str,i64,datetime[ms],datetime[ms],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""cKcuEmNzNF2""","""Mali""",1,1960-09-22 00:00:00,null,"""cKcuEmNzNF2""","""Mali""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""{""type"": ""Polygon"", ""coordinat…"
"""B3EA5FksXh3""","""Mali_del""",1,1923-11-30 00:00:00,null,"""B3EA5FksXh3""","""Mali_del""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""K2bbWqZzyrT""","""Bamako""",2,1960-09-22 00:00:00,null,"""cKcuEmNzNF2""","""Mali""","""K2bbWqZzyrT""","""Bamako""",null,null,null,null,null,null,null,null,null,null,null,null,null
"""b8cPdByQrSG""","""Gao""",2,1960-09-22 00:00:00,null,"""cKcuEmNzNF2""","""Mali""","""b8cPdByQrSG""","""Gao""",null,null,null,null,null,null,null,null,null,null,null,null,"""{""type"": ""Polygon"", ""coordinat…"
"""GTkaSxaspwx""","""Koulikoro""",2,1960-09-22 00:00:00,null,"""cKcuEmNzNF2""","""Mali""","""GTkaSxaspwx""","""Koulikoro""",null,null,null,null,null,null,null,null,null,null,null,null,"""{""type"": ""Polygon"", ""coordinat…"


## And the services

In [7]:
des = get_data_elements(dhis)

In [8]:
des = des.drop("value_type")

In [9]:
des.head()

id,name
str,str
"""ZadItokqlHj""","""A. Population a"""
"""l4h0CAPfWNY""","""Autorité de gestion"""
"""wImika3f8ds""","""BAQ Investissement"""
"""PVQmOpt9mOa""","""Baq performance manuel"""
"""Lhi3NhbG9Hz""","""BAQ Recrutement"""


## Load the things

In [10]:
ous = pl.read_parquet(
    "/home/jovyan/workspace/pipelines/initialize_vbr/data/inconsistencies/ous_model_june.parquet"
)
services = pl.read_parquet(
    "/home/jovyan/workspace/pipelines/initialize_vbr/data/inconsistencies/services_model_june.parquet"
).filter(pl.col("quarter").is_in(["2025Q2","2025Q3","2025Q4","2026Q1"]))

In [105]:
quant_data = pl.read_csv(
    "/home/jovyan/workspace/pipelines/initialize_vbr/data/quantity_data/model_june_cleaned.csv"
)
quant_data_dirty = pl.read_csv(
    "/home/jovyan/workspace/pipelines/initialize_vbr/data/quantity_data/model_june_not_cleaned.csv"
)

In [45]:
ous_payment_with_data = quant_data.filter(
    (pl.col("quarter").is_in(["2025Q2", "2025Q3", "2025Q4", "2026Q1"]))
    & (pl.col("dec").is_not_null())
    & (pl.col("val").is_not_null())
    & (pl.col("payment_mode").is_not_null())
).select(["ou", "payment_mode"]).unique()

In [108]:
quant_data_dirty.filter((pl.col("payment_mode").str.contains("paiement_dispensaire")) & (pl.col("month").str.contains("2025Q3"))).sort(by="service")

ou,month,service,dec,val,tarif,payment_mode,contract_start_date,contract_end_date,level_1_name,level_2_name,level_3_name,level_4_name,level_5_name,level_6_name,level_7_name,level_1_uid,level_2_uid,level_3_uid,level_4_uid,level_5_uid,level_6_uid,level_7_uid,level,taux_validation,ecart_dec_val,gain_verif,subside_sans_verification,subside_avec_verification,quarter
str,str,str,f64,f64,f64,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,f64,f64,f64,f64,f64,str


## Inconsistencies in the ous

In [11]:
ous.head()

payment_mode,"Valide, according to Hesabu","Valide, according to Dataset",Contracts,"Declare, according to Hesabu","Declare, according to Dataset","Valide: In Hesabu, not in Dataset","Valide: In Dataset, not in Hesabu","Valide: In Hesabu or in the Dataset, not in the Contracts","Declare: In Hesabu, not in Dataset","Declare: In Dataset, not in Hesabu","Declare: In Dataset or in the Hesabu, not in the Contracts","Hesabu: In Valide, not in Declare","Hesabu: In Declare, not in Valide","Dataset: In Valide, not in Declare","Dataset: In Declare, not in Valide"
str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[null],list[null],list[str],list[str]
"""paiement_pma""","[""ZdmA07H7tU8"", ""jsP1yc2jM3m"", … ""Ddoy2FaNEpl""]","[""ZdmA07H7tU8"", ""jsP1yc2jM3m"", … ""Ddoy2FaNEpl""]","[""QzzGgSt7xzD"", ""GWnlOrO76RC"", … ""iAss1p8eQ7o""]","[""ZdmA07H7tU8"", ""jsP1yc2jM3m"", … ""Ddoy2FaNEpl""]","[""ZdmA07H7tU8"", ""jsP1yc2jM3m"", … ""Ddoy2FaNEpl""]","[""gkQCyELPmPF"", ""XxA2rrTindX""]","[""CEJ87WBlvA5"", ""iHYkb6h8OPt"", … ""ngmfgkGgZUa""]","[""J8vq3EoxcTC"", ""TPBssMRPs97"", … ""ngmfgkGgZUa""]","[""gkQCyELPmPF"", ""XxA2rrTindX""]","[""iHYkb6h8OPt"", ""U6N0NyB4UFX"", … ""wXtlnE9WpiS""]","[""J8vq3EoxcTC"", ""TPBssMRPs97"", … ""ngmfgkGgZUa""]",[],[],"[""sVFqRLWwOzp"", ""WRcK01zDuy9"", … ""Tfa7NXVBncF""]","[""U6N0NyB4UFX"", ""knCps0Cta3j"", … ""wXtlnE9WpiS""]"
"""paiement_pca""","[""Mx7waSWhJ9Q"", ""pq9bZ5x438C"", … ""RJ8cdHFEXhk""]","[""Mx7waSWhJ9Q"", ""pq9bZ5x438C"", … ""RJ8cdHFEXhk""]","[""QzzGgSt7xzD"", ""GWnlOrO76RC"", … ""iAss1p8eQ7o""]","[""Mx7waSWhJ9Q"", ""pq9bZ5x438C"", … ""RJ8cdHFEXhk""]","[""Mx7waSWhJ9Q"", ""pq9bZ5x438C"", … ""RJ8cdHFEXhk""]",[],"[""Sg27LlS9gqS"", ""DEjywbgNgEA""]",[],[],"[""Sg27LlS9gqS"", ""DEjywbgNgEA""]",[],[],[],[],[]
"""paiement_site""","[""CFwf7AW8c08"", ""Is61QHoGg2X"", … ""bt8kXzh8gn1""]","[""CFwf7AW8c08"", ""Is61QHoGg2X"", … ""bt8kXzh8gn1""]","[""QzzGgSt7xzD"", ""GWnlOrO76RC"", … ""iAss1p8eQ7o""]","[""CFwf7AW8c08"", ""Is61QHoGg2X"", … ""bt8kXzh8gn1""]","[""CFwf7AW8c08"", ""Is61QHoGg2X"", … ""bt8kXzh8gn1""]","[""JU0oVkts2fZ"", ""dszQBduxEU1"", … ""wXtlnE9WpiS""]","[""wlBelEmOmpj"", ""veAxYx9LoH2"", … ""KQrTs7pO2SI""]","[""dFmfxOdYKib"", ""wuYaVja7ZgT"", … ""wXtlnE9WpiS""]","[""JU0oVkts2fZ"", ""dszQBduxEU1"", … ""wXtlnE9WpiS""]","[""wlBelEmOmpj"", ""veAxYx9LoH2"", … ""KQrTs7pO2SI""]","[""JU0oVkts2fZ"", ""dszQBduxEU1"", … ""mHpeKFecNrJ""]",[],[],"[""Vav1PFATiSt"", ""D9gTr2JsvyS"", … ""E8lJrN9xXYC""]","[""wdjVWyTOvhz"", ""aN2R0yQyfni"", … ""iQ4mmYoRxGV""]"
"""paiement_prive""","[""Aej6suahktX"", ""WdLrCzPh7RD"", … ""iOwgB5Gjjnx""]","[""Aej6suahktX"", ""WdLrCzPh7RD"", … ""iOwgB5Gjjnx""]","[""QzzGgSt7xzD"", ""GWnlOrO76RC"", … ""iAss1p8eQ7o""]","[""Aej6suahktX"", ""WdLrCzPh7RD"", … ""iOwgB5Gjjnx""]","[""Aej6suahktX"", ""WdLrCzPh7RD"", … ""iOwgB5Gjjnx""]","[""NPgGjX9JoJT"", ""CMq6UOeu6DP"", … ""PMKZjQOLfpO""]","[""c2jHNmSNyNS"", ""lDJEJHNRg2E"", … ""iKhQflQKkld""]","[""bILHP3coz1I"", ""HS8us8xlT2e"", … ""sxvN4taJcQC""]","[""NPgGjX9JoJT"", ""CMq6UOeu6DP"", … ""PMKZjQOLfpO""]","[""c2jHNmSNyNS"", ""i3JA5PIPvEd"", … ""iKhQflQKkld""]","[""bILHP3coz1I"", ""HS8us8xlT2e"", … ""sxvN4taJcQC""]",[],[],"[""lDJEJHNRg2E"", ""BVNP1OEI6Pa"", … ""uQxitIPNALG""]","[""YzVCQyBgTzD"", ""rpSea3TRrkG""]"
"""paiement_maternite""","[""yWdYav6Bujs"", ""LoITEQJBwbV"", … ""twOFJIQkegI""]","[""Is61QHoGg2X"", ""LoITEQJBwbV"", … ""aH5cEVrC7dN""]","[""QzzGgSt7xzD"", ""GWnlOrO76RC"", … ""iAss1p8eQ7o""]","[""yWdYav6Bujs"", ""LoITEQJBwbV"", … ""twOFJIQkegI""]","[""Is61QHoGg2X"", ""LoITEQJBwbV"", … ""aH5cEVrC7dN""]","[""XzItvJPjgDL"", ""M4VwEsXYpgS"", … ""eRdbKO4p6sa""]","[""G1fI1mvEkH5"", ""HJ5m0FRGc6D"", … ""FHUy9dss4wE""]","[""dgoBWvX6o9O"", ""TqdyXNhTay2"", … ""OXdMYPMzEg8""]","[""XzItvJPjgDL"", ""nOmqIkT1EGS"", … ""eRdbKO4p6sa""]","[""G1fI1mvEkH5"", ""HJ5m0FRGc6D"", … ""FHUy9dss4wE""]","[""dgoBWvX6o9O"", ""TqdyXNhTay2"", … ""OXdMYPMzEg8""]",[],[],"[""nOmqIkT1EGS""]",[]


What I want to do is create a table in SuperSet where I can visualize this information:
- A column that has the organisational unit
- A column that has the payment mode
- A column that has the source (hesabu / dataset / contracts)
- A column that has the declared and validated with a yes or a no.

The goal is that in this table, the client can filter very easily to see the incoherences...

In [12]:
relevant_ous = ous.select(
    [
        "payment_mode",
        "Valide, according to Hesabu",
        "Valide, according to Dataset",
        "Contracts",
        "Declare, according to Hesabu",
        "Declare, according to Dataset",
    ]
)

In [13]:
id_col = "payment_mode"
value_cols = [c for c in relevant_ous.columns if c != id_col]

long_df = (
    relevant_ous.unpivot(
        index=id_col, on=value_cols, variable_name="raw_column", value_name="org_units"
    )
    .filter(pl.col("org_units").is_not_null())
    .explode("org_units")
    .rename({"org_units": "org_unit"})
)

In [14]:
mapped_df = (
    long_df
    .with_columns(
        pl.when(pl.col("raw_column") == "Declare, according to Hesabu")
          .then(pl.lit(["declare_hesabu"]))
        .when(pl.col("raw_column") == "Valide, according to Hesabu")
          .then(pl.lit(["valide_hesabu"]))
        .when(pl.col("raw_column") == "Declare, according to Dataset")
          .then(pl.lit(["declare_dataset"]))
        .when(pl.col("raw_column") == "Valide, according to Dataset")
          .then(pl.lit(["valide_dataset"]))
        .when(pl.col("raw_column") == "Contracts")
          .then(pl.lit(["declare_contracts", "valide_contracts"]))  # duplicated
        .otherwise(pl.lit([]))
        .alias("indicator")
    ))

In [15]:
exploded_df = mapped_df.drop("raw_column").explode("indicator")
final_df = (
    exploded_df.with_columns(pl.lit(True).alias("present"))
    .pivot(
        index=["org_unit", "payment_mode"],
        on="indicator",
        values="present",
    )
    .fill_null(False)
)

## Merge the things and do some extra cleaning
For the level_1_name = "Mali_del", the things are never in the dataset -- this makes sense and should probably be raised to the client, who might want to update the contracts and the hesabu configs. (note that there are some cases in which the no-longer-used organisationalUnit is still in Hesabu, still in the contracts, or still in both.

I have also filtered to only include the organisationUnits that actually have data. There are still some of them that are not in hesabu, not in the contracts, or not in any of them. There is also difference between being assigned to the valide and to the declare (which I find very worrying).

In [64]:
merged = final_df.join(ous_dhis2, left_on="org_unit", right_on="id", how="left")

In [65]:
merged.filter(pl.col("level_1_name") == "Mali_del").select(
    [
        "valide_dataset",
        "declare_dataset",
        "valide_hesabu",
        "declare_hesabu",
        "valide_contracts",
        "declare_contracts",
        
        
    ]
).unique()

valide_dataset,declare_dataset,valide_hesabu,declare_hesabu,valide_contracts,declare_contracts
bool,bool,bool,bool,bool,bool
false,false,false,false,true,true
false,false,true,true,true,true
false,false,true,true,false,false


In [66]:
ous_really_cleaned = merged.filter(pl.col("level_1_name") != "Mali_del")

In [67]:
ous_really_cleaned = ous_really_cleaned.join(ous_payment_with_data, left_on = ["org_unit", "payment_mode"], right_on = ["ou", "payment_mode"], how="inner")

In [68]:
ous_really_cleaned.select(
    [
        "valide_dataset",
        "declare_dataset",
        "valide_hesabu",
        "declare_hesabu",
        "valide_contracts",
        "declare_contracts",
        
        
    ]
).unique()

valide_dataset,declare_dataset,valide_hesabu,declare_hesabu,valide_contracts,declare_contracts
bool,bool,bool,bool,bool,bool
true,true,false,false,false,false
true,true,true,true,false,false
false,true,false,false,true,true
true,true,true,true,true,true
true,true,false,false,true,true
true,false,false,false,true,true


In [52]:
ous_really_cleaned = merged_sel.filter(pl.col("level_1_name") != "Mali_del")

## Save into a database

In [54]:
ous_inconsistents = ous_really_cleaned.to_pandas()

In [55]:
if save_in_db:
    engine = create_engine(environ["WORKSPACE_DATABASE_URL"])
    database_name = "inconsistencies_ous"
    ous_inconsistents.to_sql(database_name, con = engine, if_exists="replace")

## Lets do some extra explorations

# Now, services

In [21]:
services.head()

payment_mode,quarter,indicators,Declare: Services not in Hesabu,Declare: List of services not in Hesabu,Declare: Percentage of nulls rows,Valide: Services not in Hesabu,Valide: List of services not in Hesabu,Valide: Percentage of nulls rows,Tarif_def: Services not in Hesabu,Tarif_def: List of services not in Hesabu,Tarif_def: Percentage of nulls rows
str,str,list[str],str,list[str],str,str,list[str],str,str,list[str],str
"""paiement_pma""","""2025Q2""","[""declare"", ""valide"", ""tarif_def""]","""3 / 36""","[""r5y4iAaiGAu"", ""WAVrIW5NEhX"", ""s5DOhQ8E7pr""]","""0.24""","""""",[],"""7.37""","""4 / 37""","[""xY9uofEZq2r"", ""hxkWry4HoRC"", … ""Wzt2uTx33UU""]",""""""
"""paiement_pma""","""2025Q3""","[""declare"", ""valide"", ""tarif_def""]","""3 / 36""","[""s5DOhQ8E7pr"", ""r5y4iAaiGAu"", ""WAVrIW5NEhX""]","""0.34""","""3 / 36""","[""AdkRTRB1Wq4"", ""r0p5VaG6ATf"", ""UcY46Hro5jU""]","""3.51""","""9 / 42""","[""b9OmkKrJevY"", ""UtDknCwMy4H"", … ""Wzt2uTx33UU""]",""""""
"""paiement_pma""","""2025Q4""","[""declare"", ""valide"", ""tarif_def""]","""4 / 37""","[""WAVrIW5NEhX"", ""s5DOhQ8E7pr"", … ""MuR2FZNizfo""]","""0.42""","""3 / 36""","[""AdkRTRB1Wq4"", ""r0p5VaG6ATf"", ""UcY46Hro5jU""]","""1.96""","""10 / 43""","[""ssXW5Cz38IB"", ""hxkWry4HoRC"", … ""Wzt2uTx33UU""]",""""""
"""paiement_pma""","""2026Q1""","[""declare"", ""valide"", ""tarif_def""]","""3 / 36""","[""WAVrIW5NEhX"", ""s5DOhQ8E7pr"", ""r5y4iAaiGAu""]","""0.21""","""3 / 36""","[""AdkRTRB1Wq4"", ""r0p5VaG6ATf"", ""UcY46Hro5jU""]","""2.31""","""10 / 43""","[""ssXW5Cz38IB"", ""b9OmkKrJevY"", … ""zNne4IcIJdF""]",""""""
"""paiement_pca""","""2025Q2""","[""declare"", ""valide"", ""tarif_def""]","""5 / 39""","[""SQb6WynojTw"", ""dYgVbkbxzWs"", … ""KdgRPphL7wN""]","""0.16""","""""",[],"""4.38""","""3 / 37""","[""eRWIQ3GZ8TY"", ""VbHGAiOBMez"", ""icoHa7HPsKj""]",""""""


With this, I want a table that has
- A column with the payment_mode
- A column with the quarter
- A column with tarif, declare and valide -- with a yes or no indicating if they are present

And then I also need, for the ones that have ["declare","valide","tarif_def"] I want to create a table that has
- A column with the payment_mode
- A column with the quarter
- A column with the service
- A column with the declare, valide and tarif, indicating if it is there or not

In [88]:
present_indicators = (
    services
    .with_columns(
        pl.col("indicators").list.contains("declare").alias("declare"),
        pl.col("indicators").list.contains("valide").alias("valide"),
        pl.col("indicators").list.contains("tarif_def").alias("tarif"),
    )
    .select(
        "payment_mode",
        "quarter",
        "declare",
        "valide",
        "tarif",
    )
)

In [89]:
present_indicators = present_indicators.with_columns(
    pl.concat_str(
        pl.col("declare").cast(pl.Int8).cast(pl.String),
        pl.col("valide").cast(pl.Int8).cast(pl.String),
        pl.col("tarif").cast(pl.Int8).cast(pl.String),
        separator="_"
    ).alias("fingerprint")
)

In [90]:
present_indicators_weird_unpivoted = (
    present_indicators
    .group_by("payment_mode")
    .agg(
        pl.col("fingerprint").n_unique().alias("n_distinct_combos"),
        pl.col("quarter").n_unique().alias("n_quarters"),
        pl.col("quarter").sort().alias("quarters_present"),
        pl.col("fingerprint").sort().alias("fingerprints"),
    )
    .filter(
        (pl.col("n_distinct_combos") > 1)  # indicators differ across quarters
        | (pl.col("n_quarters") < 4)  # or some quarters are missing
    )
    .sort("payment_mode")
)
weird_indicators = present_indicators_weird_unpivoted.select(pl.col("payment_mode")).unique().to_series().to_list()

In [96]:
present_indicators_weird = present_indicators.filter(pl.col("payment_mode").is_in(weird_indicators)).drop("fingerprint")

In [98]:
present_indicators_weird = present_indicators_weird.to_pandas()

In [99]:
if save_in_db:
    engine = create_engine(environ["WORKSPACE_DATABASE_URL"])
    database_name = "inconsistencies_indicators"
    present_indicators_weird.to_sql(database_name, con = engine, if_exists="replace")

## And the other one

In [25]:
services_present_1 = services.filter(
    pl.col("indicators") == ["declare", "valide", "tarif_def"]
).select(
    [
        "payment_mode",
        "quarter",
        "Declare: List of services not in Hesabu",
        "Tarif_def: List of services not in Hesabu",
        "Valide: List of services not in Hesabu"
    ]
)

In [26]:
services_present_1

payment_mode,quarter,Declare: List of services not in Hesabu,Tarif_def: List of services not in Hesabu,Valide: List of services not in Hesabu
str,str,list[str],list[str],list[str]
"""paiement_pma""","""2025Q2""","[""r5y4iAaiGAu"", ""WAVrIW5NEhX"", ""s5DOhQ8E7pr""]","[""xY9uofEZq2r"", ""hxkWry4HoRC"", … ""Wzt2uTx33UU""]",[]
"""paiement_pma""","""2025Q3""","[""s5DOhQ8E7pr"", ""r5y4iAaiGAu"", ""WAVrIW5NEhX""]","[""b9OmkKrJevY"", ""UtDknCwMy4H"", … ""Wzt2uTx33UU""]","[""AdkRTRB1Wq4"", ""r0p5VaG6ATf"", ""UcY46Hro5jU""]"
"""paiement_pma""","""2025Q4""","[""WAVrIW5NEhX"", ""s5DOhQ8E7pr"", … ""MuR2FZNizfo""]","[""ssXW5Cz38IB"", ""hxkWry4HoRC"", … ""Wzt2uTx33UU""]","[""AdkRTRB1Wq4"", ""r0p5VaG6ATf"", ""UcY46Hro5jU""]"
"""paiement_pma""","""2026Q1""","[""WAVrIW5NEhX"", ""s5DOhQ8E7pr"", ""r5y4iAaiGAu""]","[""ssXW5Cz38IB"", ""b9OmkKrJevY"", … ""zNne4IcIJdF""]","[""AdkRTRB1Wq4"", ""r0p5VaG6ATf"", ""UcY46Hro5jU""]"
"""paiement_pca""","""2025Q2""","[""SQb6WynojTw"", ""dYgVbkbxzWs"", … ""KdgRPphL7wN""]","[""eRWIQ3GZ8TY"", ""VbHGAiOBMez"", ""icoHa7HPsKj""]",[]
…,…,…,…,…
"""paiement_cabsf""","""2026Q1""",[],[],[]
"""paiement_cabmed""","""2025Q2""",[],[],[]
"""paiement_cabmed""","""2025Q3""",[],[],[]


In [27]:
services_present_2 = (
    services_present_1
    .with_columns(
        pl.concat_list(
            pl.col("Declare: List of services not in Hesabu"),
            pl.col("Valide: List of services not in Hesabu"),
            pl.col("Tarif_def: List of services not in Hesabu"),
        )
        .list.unique()
        .alias("service")
    )
    .explode("service")
)
services_present_2 = services_present_2.filter(pl.col("service").is_not_null())

In [28]:
services_present_3 = (
    services_present_2
    .with_columns(
        pl.col("Declare: List of services not in Hesabu")
        .list.contains(pl.col("service"))
        .alias("declare"),

        pl.col("Valide: List of services not in Hesabu")
        .list.contains(pl.col("service"))
        .alias("valide"),

        pl.col("Tarif_def: List of services not in Hesabu")
        .list.contains(pl.col("service"))
        .alias("tarif"),
    )
    .select(
        "payment_mode",
        "quarter",
        "service",
        "declare",
        "valide",
        "tarif",
    )
)

In [29]:
services_present_4 = services_present_3.join(des, left_on="service", right_on="id")

In [30]:
services_present_5 = services_present_4.with_columns(
    pl.col("name")
    .str.replace("- déclaré ", "")
    .str.replace("- validé ", "")
    .str.replace("- tarif déf ", "")
    .str.replace("- tarif ", "")
    .str.replace("- tarif déf. ", "")
    .str.replace("- tarif_p ", "")
).drop("service")

In [31]:
services_present_6 = (
    services_present_5
    .group_by(["payment_mode", "quarter", "name"])
    .agg([
        pl.col("declare").any().alias("declare"),
        pl.col("valide").any().alias("valide"),
        pl.col("tarif").any().alias("tarif"),
    ])
)
services_present_6 = services_present_6.to_pandas()

In [32]:
if save_in_db:
    engine = create_engine(environ["WORKSPACE_DATABASE_URL"])
    database_name = "inconsistencies_services"
    services_present_6.to_sql(database_name, con = engine, if_exists="replace")